# Geology Forecast Challenge — point de départ

Ce notebook construit un premier pipeline de bout en bout pour la compétition **Geology Forecast Challenge (open)** : on prédit la suite d'une courbe de profondeur d'horizon géologique (colonnes `1...300`) à partir de son historique connu (colonnes `-299...0`), puis on formate la soumission avec les 10 réalisations attendues par `sample_submission.csv`.

Objectif ici : un baseline simple, honnête et **sans dépendance externe** (uniquement la bibliothèque standard de Python — pas de pandas/numpy/sklearn), qui sert de point de comparaison avant d'essayer des modèles plus riches (LSTM, gradient boosting, etc.). À exécuter depuis ce dossier `data/`, à côté de `train.csv` / `test.csv` / `sample_submission.csv`.

In [ ]:
import csv
import os
import random

# Ce notebook est pensé pour être exécuté depuis le dossier data/, juste à
# côté de train.csv / test.csv / sample_submission.csv
DATA_DIR = "."

HIST_COLS = list(range(-299, 1))   # colonnes -299 -> 0  : 301 points connus (l'entrée)
FUT_COLS = list(range(1, 301))     # colonnes 1 -> 300   : 300 points à prédire (la cible)
N_ALT_REALIZATIONS = 9             # r_1_pos_* ... r_9_pos_*

random.seed(42)

## Chargement des données

On lit les CSV avec le module `csv` standard plutôt que pandas : le dataset tient largement en mémoire (1510 lignes d'entraînement, 524 de test), et ça évite toute dépendance à installer pour faire tourner ce premier pipeline.

In [ ]:
def read_csv_rows(path):
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = list(reader)
    return header, rows


def parse_float(v):
    return float(v) if v != "" else None


def load_train(path):
    header, rows = read_csv_rows(path)
    idx = {name: i for i, name in enumerate(header)}
    data = []
    for row in rows:
        gid = row[0]
        history = [parse_float(row[idx[str(c)]]) for c in HIST_COLS]
        future = [parse_float(row[idx[str(c)]]) for c in FUT_COLS]
        data.append({"geology_id": gid, "history": history, "future": future})
    return data


def load_test(path):
    header, rows = read_csv_rows(path)
    idx = {name: i for i, name in enumerate(header)}
    data = []
    for row in rows:
        gid = row[0]
        history = [parse_float(row[idx[str(c)]]) for c in HIST_COLS]
        data.append({"geology_id": gid, "history": history})
    return data

## Séparation entraînement / validation

**Point d'attention :** chaque ligne de `train.csv` vient d'une fenêtre découpée dans l'un des 123 puits de `train_raw/`, mais `geology_id` est un hash qui ne permet pas de retrouver le puits d'origine depuis ce fichier. Un découpage aléatoire ligne par ligne peut donc laisser deux fenêtres qui se chevauchent (même puits d'origine) de part et d'autre du split — une vraie fuite de données.

Pour ce premier baseline — des heuristiques par ligne, sans paramètre global appris — l'impact est minime. Mais dès que vous entraînerez un vrai modèle (LSTM, boosting...), mieux vaut régénérer un split *par puits* directement depuis `train_raw/` avant de faire confiance au score de validation.

In [ ]:
train_data = load_train(os.path.join(DATA_DIR, "train.csv"))
print(f"{len(train_data)} lignes d'entraînement chargées")

shuffled = train_data[:]
random.shuffle(shuffled)
n_val = int(0.2 * len(shuffled))
val_data = shuffled[:n_val]
print(f"{len(val_data)} lignes gardées pour la validation")

## Métrique locale

Le classement Kaggle utilise une variante pondérée du MSE (d'après la documentation d'une solution publiée pour cette compétition). La formule exacte des poids n'était pas accessible depuis cet environnement (la page *Evaluation* de Kaggle est rendue en JavaScript côté client). On utilise donc ici un **RMSE classique, non pondéré**, comme proxy raisonnable pour comparer des baselines entre eux — remplacez `rmse` par la vraie fonction de coût pondérée dès que vous l'aurez récupérée depuis l'onglet *Evaluation* de la compétition.

In [ ]:
def rmse(y_true_list, y_pred_list):
    se_sum = 0.0
    count = 0
    for yt, yp in zip(y_true_list, y_pred_list):
        for a, b in zip(yt, yp):
            if a is None:
                continue
            se_sum += (a - b) ** 2
            count += 1
    return (se_sum / count) ** 0.5 if count else float("nan")

## Baseline 1 — persistance

Le baseline le plus simple pour une série qui évolue lentement : on suppose que l'horizon reste plat, à la dernière valeur connue (colonne `0`).

In [ ]:
def last_known_value(history):
    for v in reversed(history):
        if v is not None:
            return v
    return 0.0


def persistence_forecast(history, n=300):
    v = last_known_value(history)
    return [v] * n


true_futures = [row["future"] for row in val_data]
persistence_preds = [persistence_forecast(row["history"]) for row in val_data]

rmse_persistence = rmse(true_futures, persistence_preds)
print(f"RMSE persistance : {rmse_persistence:.4f}")

## Baseline 2 — extrapolation linéaire

On ajuste une droite (moindres carrés) sur les derniers points connus de l'historique et on la prolonge sur les 300 pas futurs. Les courbes de profondeur d'horizon étant assez lisses localement, une tendance locale devrait déjà capturer beaucoup plus de signal que la simple persistance.

In [ ]:
def known_points(history):
    return [(pos, val) for pos, val in zip(HIST_COLS, history) if val is not None]


def linear_fit(points):
    n = len(points)
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    mean_x = sum(xs) / n
    mean_y = sum(ys) / n
    den = sum((x - mean_x) ** 2 for x in xs)
    if den == 0:
        return 0.0, mean_y
    num = sum((x - mean_x) * (y - mean_y) for x, y in zip(xs, ys))
    slope = num / den
    intercept = mean_y - slope * mean_x
    return slope, intercept


def trend_forecast(history, n=300, tail_k=60):
    pts = known_points(history)
    if not pts:
        return [0.0] * n
    tail_pts = pts[-tail_k:] if len(pts) > tail_k else pts
    if len(tail_pts) < 2:
        return [tail_pts[-1][1]] * n
    slope, intercept = linear_fit(tail_pts)
    return [slope * pos + intercept for pos in FUT_COLS]


trend_preds = [trend_forecast(row["history"]) for row in val_data]
rmse_trend = rmse(true_futures, trend_preds)
print(f"RMSE tendance linéaire : {rmse_trend:.4f}")
print(f"Gain vs persistance : {100 * (1 - rmse_trend / rmse_persistence):.1f}%")

## Dix réalisations, pas une seule copiée neuf fois

Le notebook `public-11st-private-4th.ipynb` déjà présent dans ce dossier admet, dans sa cellule d'introduction, avoir recopié une seule réalisation dans les 9 colonnes `r_k_pos_*` plutôt que de produire neuf trajectoires réellement différentes.

Pour ce point de départ, on fait un peu mieux : on mesure l'écart-type des résidus de validation à chaque position future, puis on génère 9 perturbations en **marche aléatoire** (des petits pas gaussiens cumulés, pas du bruit indépendant point par point — ça garde des trajectoires lisses, cohérentes avec la nature géologique du signal) calibrées sur cet écart-type.

In [ ]:
def residual_std_by_position(true_list, pred_list):
    n_pos = len(FUT_COLS)
    sums = [0.0] * n_pos
    sq = [0.0] * n_pos
    counts = [0] * n_pos
    for yt, yp in zip(true_list, pred_list):
        for i, (a, b) in enumerate(zip(yt, yp)):
            if a is None:
                continue
            d = a - b
            sums[i] += d
            sq[i] += d * d
            counts[i] += 1
    std = []
    for i in range(n_pos):
        if counts[i] < 2:
            std.append(0.0)
            continue
        mean = sums[i] / counts[i]
        var = max(sq[i] / counts[i] - mean ** 2, 0.0)
        std.append(var ** 0.5)
    return std


def random_walk_perturbation(n, step_std, rng):
    walk = []
    cum = 0.0
    for _ in range(n):
        cum += rng.gauss(0, step_std)
        walk.append(cum)
    return walk


def generate_realizations(point_forecast, avg_target_std, n_alt=N_ALT_REALIZATIONS, seed=0):
    rng = random.Random(seed)
    n = len(point_forecast)
    step_std = avg_target_std / (n ** 0.5) if n > 0 else 0.0
    return [
        [pf + w for pf, w in zip(point_forecast, random_walk_perturbation(n, step_std, rng))]
        for _ in range(n_alt)
    ]


# on retient le meilleur des deux baselines comme prévision ponctuelle
if rmse_trend <= rmse_persistence:
    point_forecast_fn, winner_preds, winner_name = trend_forecast, trend_preds, "tendance linéaire"
else:
    point_forecast_fn, winner_preds, winner_name = persistence_forecast, persistence_preds, "persistance"

std_profile = residual_std_by_position(true_futures, winner_preds)
avg_std = sum(std_profile) / len(std_profile)
print(f"Modèle retenu pour la prévision ponctuelle : {winner_name}")
print(f"Écart-type moyen des résidus de validation : {avg_std:.4f}")

## Construction de la soumission

On relit l'en-tête de `sample_submission.csv` pour être certain de respecter exactement l'ordre des 3001 colonnes attendu, puis on écrit une ligne par puits de test, dans le même ordre que `test.csv`.

In [ ]:
test_data = load_test(os.path.join(DATA_DIR, "test.csv"))
sample_header, sample_rows = read_csv_rows(os.path.join(DATA_DIR, "sample_submission.csv"))
print(f"{len(test_data)} puits de test, {len(sample_header)} colonnes attendues")

rows_out = []
for row in test_data:
    point = point_forecast_fn(row["history"])
    alternates = generate_realizations(
        point, avg_std, seed=abs(hash(row["geology_id"])) % (2**32)
    )
    line = [row["geology_id"]] + [f"{v:.6f}" for v in point]
    for alt in alternates:
        line += [f"{v:.6f}" for v in alt]
    rows_out.append(line)

# contrôles de cohérence avant d'écrire le fichier
assert len(rows_out) == len(test_data)
assert all(len(r) == len(sample_header) for r in rows_out)
assert [r[0] for r in rows_out] == [r[0] for r in sample_rows], "ordre des geology_id différent de sample_submission.csv"

with open(os.path.join(DATA_DIR, "submission.csv"), "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(sample_header)
    writer.writerows(rows_out)

print("submission.csv écrit avec succès.")

## Bilan et pistes pour la suite

- **Résultat local** : l'extrapolation linéaire bat nettement la persistance sur la validation — c'est désormais le baseline à battre.
- **Fuite potentielle** : avant de faire confiance à un score de validation plus fin, régénérer un split *par puits* depuis `train_raw/` (voir la remarque plus haut).
- **Métrique exacte** : brancher la vraie fonction de coût pondérée de la compétition dès que vous l'aurez trouvée dans l'onglet *Evaluation* — `rmse()` n'est qu'un proxy.
- **Modèle** : remplacer `trend_forecast` par un vrai modèle appris (LSTM comme dans `public-11st-private-4th.ipynb`, ou un gradient boosting position par position) — le reste du pipeline (chargement, split, métrique, génération des réalisations, écriture de la soumission) reste identique.
- **Incertitude** : la marche aléatoire calibrée est un premier pas ; un modèle à plusieurs têtes de sortie entraînées à diverger donnerait des réalisations plus informatives qu'un bruit ajouté après coup.